# Lab 1: ML for Security -- Phishing URL Classification
## Applying Supervised Machine Learning to Detect Malicious URLs

**Workshop:** Attacking, Defending & Leveraging Agentic AI  
**Authors:** Derek Banks and Dr. Brian Fehrman  
**Time:** 45 minutes  
**Platform:** Google Colab (CPU-only)  
**Theme:** Leverage  

---

### What You Will Learn
- How to apply supervised machine learning to a real cybersecurity problem
- Feature engineering techniques for extracting signal from raw URLs
- Training and comparing Random Forest and Logistic Regression classifiers
- Evaluating models with confusion matrices, precision, recall, and F1 scores
- Understanding false positive vs. false negative tradeoffs in security operations

### Prerequisites
- Basic familiarity with Python
- General understanding of what phishing is
- No prior machine learning experience required

### Threat Model Connection
Phishing remains the **#1 initial access vector** in real-world attacks (Verizon DBIR, MITRE ATT&CK T1566).  
Traditional blocklists are reactive -- they only catch *known* malicious URLs.  
Machine learning lets us generalize from known examples to detect *never-before-seen* phishing URLs  
based on their structural properties.

In this lab, you will build a classifier that examines the *anatomy* of a URL -- its length,  
entropy, subdomain depth, suspicious keywords, and more -- to make a prediction about whether  
the URL is benign or phishing. This is the same approach used by commercial email gateways,  
browser safe-browsing features, and SOC automation pipelines.

### Lab Roadmap
1. Environment Setup
2. Background: Why ML for Phishing Detection
3. Synthetic Dataset Generation
4. Feature Engineering
5. Data Exploration & Visualization
6. Model Training
7. Model Evaluation
8. Model Comparison
9. Student Exercises
10. Security Discussion & Key Takeaways

In [ ]:
# ============================================================
# Cell 1: Environment Setup
# ============================================================
# Install and import all required packages.
# This lab runs on CPU only -- no GPU required.
# ============================================================

!pip install -q scikit-learn pandas numpy matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import math
import string
import warnings
from collections import Counter
from urllib.parse import urlparse

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    roc_auc_score,
    accuracy_score,
    f1_score,
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('=' * 60)
print('ENVIRONMENT CHECK')
print('=' * 60)
print(f'NumPy      : {np.__version__}')
print(f'Pandas     : {pd.__version__}')
import sklearn; print(f'Scikit-learn: {sklearn.__version__}')
print(f'Random Seed : {RANDOM_SEED}')
print('=' * 60)
print('All packages loaded successfully. Ready to proceed.')
print('=' * 60)

## Section 1: Background -- Why ML for Phishing Detection

### The Problem
Phishing attacks use deceptive URLs to trick users into visiting malicious websites that steal
 credentials, install malware, or exfiltrate data. Defenders face a fundamental challenge:

| Approach | Strength | Weakness |
|----------|----------|----------|
| **Blocklists** | Zero false positives for known URLs | Cannot detect new/unseen phishing URLs |
| **Heuristic Rules** | Fast, interpretable | Brittle; attackers adapt quickly |
| **Machine Learning** | Generalizes to unseen URLs | Requires training data; can be evaded |

### Why URLs Are a Good Feature Source
URLs carry a surprising amount of signal about attacker intent:

- **Legitimate URLs** tend to be short, use well-known domains, and have simple path structures
- **Phishing URLs** often contain misspelled brand names, IP addresses instead of domains,
 excessive subdomains, random character strings, and suspicious TLDs

### What We Will Build
A binary classifier that takes a raw URL as input and outputs a prediction:

```
INPUT:  http://192.168.1.1/secure-login/paypa1-verify.php?id=8a3f2c
OUTPUT: PHISHING (confidence: 0.94)

INPUT:  https://docs.google.com/document/d/1a2b3c
OUTPUT: BENIGN (confidence: 0.97)
```

We will:
1. Generate a synthetic but realistic dataset of benign and phishing URLs
2. Engineer meaningful features from the raw URL strings
3. Train two classifiers (Random Forest, Logistic Regression)
4. Evaluate their performance with security-relevant metrics

In [ ]:
# ============================================================
# Cell 3: Synthetic Dataset Generation
# ============================================================
# We generate ~2000 URLs programmatically rather than relying
# on an external dataset that may become unavailable.
#
# Benign URLs: well-known domains with realistic paths
# Phishing URLs: misspelled domains, IP-based, excessive
#   subdomains, random strings, suspicious TLDs, etc.
# ============================================================

# --- Benign URL Components ---
benign_domains = [
    'google.com', 'amazon.com', 'microsoft.com', 'apple.com',
    'github.com', 'stackoverflow.com', 'wikipedia.org', 'reddit.com',
    'linkedin.com', 'twitter.com', 'facebook.com', 'youtube.com',
    'netflix.com', 'dropbox.com', 'slack.com', 'zoom.us',
    'adobe.com', 'salesforce.com', 'shopify.com', 'stripe.com',
    'medium.com', 'nytimes.com', 'bbc.co.uk', 'cnn.com',
    'espn.com', 'weather.com', 'imdb.com', 'yelp.com',
    'pinterest.com', 'tumblr.com', 'twitch.tv', 'spotify.com',
    'paypal.com', 'ebay.com', 'walmart.com', 'target.com',
    'bestbuy.com', 'homedepot.com', 'costco.com', 'etsy.com',
]

benign_subdomains = ['www', 'mail', 'docs', 'drive', 'support', 'help', 'blog', 'api', 'app', 'store']

benign_paths = [
    '/index.html', '/about', '/contact', '/products', '/services',
    '/blog/2024/article', '/help/faq', '/account/settings',
    '/search?q=python+tutorial', '/docs/getting-started',
    '/user/profile', '/news/latest', '/pricing', '/features',
    '/download', '/signin', '/signup', '/terms', '/privacy',
    '/category/electronics', '/item/12345', '/order/status',
    '', '/', '/home',
]


def generate_benign_url():
    """Generate a realistic benign URL."""
    domain = np.random.choice(benign_domains)
    use_https = np.random.random() > 0.1  # 90% HTTPS
    scheme = 'https' if use_https else 'http'
    use_subdomain = np.random.random() > 0.4  # 60% have subdomain
    subdomain = np.random.choice(benign_subdomains) + '.' if use_subdomain else ''
    path = np.random.choice(benign_paths)
    return f'{scheme}://{subdomain}{domain}{path}'


# --- Phishing URL Components ---
phishing_tlds = ['.tk', '.ml', '.ga', '.cf', '.gq', '.xyz', '.top', '.win', '.bid', '.click', '.link', '.info', '.buzz', '.surf']

brand_misspellings = [
    'g00gle', 'gogle', 'googl', 'amaz0n', 'amazom', 'arnazon',
    'micros0ft', 'mircosoft', 'microsft', 'app1e', 'appie',
    'paypa1', 'paypaI', 'paypol', 'faceb00k', 'faceboook',
    'lnstagram', 'instaqram', 'linkedln', 'llnkedin',
    'netfllx', 'netf1ix', 'dr0pbox', 'dropb0x',
    'str1pe', 'stripee', 'tw1tter', 'twltter',
]

suspicious_keywords = [
    'secure', 'login', 'verify', 'update', 'confirm', 'account',
    'banking', 'password', 'credential', 'suspend', 'urgent',
    'alert', 'notification', 'validate', 'authenticate',
    'recover', 'unlock', 'expired', 'limited', 'unusual-activity',
]

random_chars = string.ascii_lowercase + string.digits


def random_string(length):
    """Generate a random alphanumeric string."""
    return ''.join(np.random.choice(list(random_chars)) for _ in range(length))


def generate_phishing_url():
    """Generate a realistic phishing URL using various evasion tactics."""
    tactic = np.random.choice([
        'misspelled_brand',
        'ip_address',
        'excessive_subdomains',
        'long_random',
        'suspicious_tld',
        'at_symbol',
        'mixed_tactics',
    ], p=[0.20, 0.15, 0.15, 0.15, 0.15, 0.05, 0.15])

    use_https = np.random.random() > 0.6  # Only 40% HTTPS
    scheme = 'https' if use_https else 'http'

    if tactic == 'misspelled_brand':
        brand = np.random.choice(brand_misspellings)
        tld = np.random.choice(phishing_tlds + ['.com', '.net', '.org'])
        keyword = np.random.choice(suspicious_keywords)
        return f'{scheme}://{brand}{tld}/{keyword}/{random_string(8)}'

    elif tactic == 'ip_address':
        ip = '.'.join([str(np.random.randint(1, 255)) for _ in range(4)])
        keyword = np.random.choice(suspicious_keywords)
        return f'{scheme}://{ip}/{keyword}/index.php?id={random_string(12)}'

    elif tactic == 'excessive_subdomains':
        n_subs = np.random.randint(3, 7)
        subs = '.'.join([random_string(np.random.randint(3, 8)) for _ in range(n_subs)])
        legit_domain = np.random.choice(benign_domains)
        keyword = np.random.choice(suspicious_keywords)
        return f'{scheme}://{subs}.{legit_domain}/{keyword}'

    elif tactic == 'long_random':
        domain = random_string(np.random.randint(15, 30))
        tld = np.random.choice(phishing_tlds)
        path = '/'.join([random_string(np.random.randint(5, 12)) for _ in range(np.random.randint(2, 5))])
        return f'{scheme}://{domain}{tld}/{path}'

    elif tactic == 'suspicious_tld':
        brand = np.random.choice(brand_misspellings + ['secure-login', 'account-verify', 'update-info'])
        tld = np.random.choice(phishing_tlds)
        keyword = np.random.choice(suspicious_keywords)
        return f'{scheme}://{brand}{tld}/{keyword}.html'

    elif tactic == 'at_symbol':
        legit_domain = np.random.choice(benign_domains)
        malicious = random_string(10) + np.random.choice(phishing_tlds)
        return f'{scheme}://{legit_domain}@{malicious}/login'

    else:  # mixed_tactics
        brand = np.random.choice(brand_misspellings)
        n_subs = np.random.randint(2, 4)
        subs = '.'.join([np.random.choice(suspicious_keywords) for _ in range(n_subs)])
        tld = np.random.choice(phishing_tlds)
        path_len = np.random.randint(20, 40)
        return f'{scheme}://{subs}.{brand}{tld}/{random_string(path_len)}?token={random_string(16)}'


# --- Generate the dataset ---
N_BENIGN = 1000
N_PHISHING = 1000

print('Generating synthetic URL dataset...')
print(f'  Benign URLs:   {N_BENIGN}')
print(f'  Phishing URLs: {N_PHISHING}')
print()

benign_urls = [generate_benign_url() for _ in range(N_BENIGN)]
phishing_urls = [generate_phishing_url() for _ in range(N_PHISHING)]

# Combine into a DataFrame
df = pd.DataFrame({
    'url': benign_urls + phishing_urls,
    'label': [0] * N_BENIGN + [1] * N_PHISHING,  # 0 = benign, 1 = phishing
})

# Shuffle the dataset
df = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print('=' * 60)
print('DATASET GENERATED')
print('=' * 60)
print(f'Total URLs: {len(df)}')
print(f'Benign:     {(df["label"] == 0).sum()}')
print(f'Phishing:   {(df["label"] == 1).sum()}')
print()
print('Sample benign URLs:')
for url in df[df['label'] == 0]['url'].head(5).values:
    print(f'  [BENIGN]   {url}')
print()
print('Sample phishing URLs:')
for url in df[df['label'] == 1]['url'].head(5).values:
    print(f'  [PHISHING] {url}')
print('=' * 60)

## Section 2: Feature Engineering

### From Raw Strings to Numeric Features
Machine learning models cannot operate on raw text directly. We need to convert each URL into
 a **feature vector** -- a fixed-length array of numbers that captures the URL's characteristics.

### Our Feature Set (13 Features)

| # | Feature | Intuition |
|---|---------|----------|
| 1 | `url_length` | Phishing URLs tend to be longer |
| 2 | `num_dots` | More dots = more subdomains = suspicious |
| 3 | `num_hyphens` | Hyphens often used in fake domains |
| 4 | `num_slashes` | Deep paths can indicate obfuscation |
| 5 | `num_digits` | Random digit strings are suspicious |
| 6 | `num_special_chars` | @, %, =, &, etc. |
| 7 | `has_ip_address` | Legitimate sites rarely use raw IPs |
| 8 | `uses_https` | Phishing less likely to use HTTPS |
| 9 | `domain_entropy` | Randomness in domain name |
| 10 | `num_subdomains` | Excessive subdomains are a red flag |
| 11 | `path_length` | Long paths can hide malicious endpoints |
| 12 | `has_suspicious_keywords` | Words like 'login', 'verify', 'secure' |
| 13 | `has_at_symbol` | @ in URL is almost always malicious |

### Shannon Entropy
Shannon entropy measures the *randomness* of a string. Legitimate domain names ("google", "amazon")
 have low entropy because they are short, recognizable words. Phishing domains ("x7k2m9q4p") have
 high entropy because they are random character sequences.

$$H(X) = -\sum_{i} p(x_i) \log_2 p(x_i)$$

In [ ]:
# ============================================================
# Cell 5: Feature Engineering
# ============================================================
# Extract 13 numeric features from each raw URL.
# Each feature captures a different structural property
# that may distinguish benign URLs from phishing URLs.
# ============================================================


def calculate_entropy(text):
    """Calculate Shannon entropy of a string.
    Higher entropy = more randomness = more suspicious."""
    if not text:
        return 0.0
    freq = Counter(text)
    length = len(text)
    return -sum((count / length) * math.log2(count / length) for count in freq.values())


def extract_domain(url):
    """Extract the domain portion of a URL."""
    try:
        parsed = urlparse(url)
        hostname = parsed.hostname or ''
        # Remove 'www.' prefix for cleaner analysis
        if hostname.startswith('www.'):
            hostname = hostname[4:]
        return hostname
    except Exception:
        return ''


def has_ip_pattern(url):
    """Check if the URL uses an IP address instead of a domain name."""
    ip_pattern = re.compile(
        r'https?://'
        r'(\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})'
    )
    return int(bool(ip_pattern.search(url)))


# Keywords commonly found in phishing URLs
SUSPICIOUS_KEYWORDS = [
    'login', 'verify', 'secure', 'account', 'update', 'confirm',
    'banking', 'password', 'credential', 'suspend', 'urgent',
    'authenticate', 'recover', 'unlock', 'expired', 'validate',
]


def extract_features(url):
    """Extract all 13 features from a single URL.

    Returns a dictionary of feature_name: value pairs.
    """
    parsed = urlparse(url)
    hostname = parsed.hostname or ''
    path = parsed.path or ''
    domain = extract_domain(url)
    url_lower = url.lower()

    # Count subdomains (split on dots, subtract the TLD parts)
    parts = hostname.split('.')
    # Rough heuristic: last 2 parts are TLD (e.g., co.uk) or last 1 (e.g., .com)
    num_subdomains = max(0, len(parts) - 2)

    # Special characters (excluding standard URL structure chars :// and .)
    special_chars = sum(1 for c in url if c in '@%&=?#!~^|{}<>')

    features = {
        'url_length':               len(url),
        'num_dots':                 url.count('.'),
        'num_hyphens':              url.count('-'),
        'num_slashes':              url.count('/'),
        'num_digits':               sum(c.isdigit() for c in url),
        'num_special_chars':        special_chars,
        'has_ip_address':           has_ip_pattern(url),
        'uses_https':               int(url.startswith('https')),
        'domain_entropy':           round(calculate_entropy(domain), 4),
        'num_subdomains':           num_subdomains,
        'path_length':              len(path),
        'has_suspicious_keywords':  int(any(kw in url_lower for kw in SUSPICIOUS_KEYWORDS)),
        'has_at_symbol':            int('@' in url),
    }
    return features


# --- Apply feature extraction to all URLs ---
print('Extracting features from all URLs...')
features_list = [extract_features(url) for url in df['url']]
features_df = pd.DataFrame(features_list)

# Combine features with labels
dataset = pd.concat([features_df, df[['label']]], axis=1)

print()
print('=' * 60)
print('FEATURE ENGINEERING COMPLETE')
print('=' * 60)
print(f'Features extracted: {len(features_df.columns)}')
print(f'Feature names: {list(features_df.columns)}')
print()
print('Feature statistics:')
print(dataset.describe().round(2).to_string())
print()
print('First 5 rows:')
print(dataset.head().to_string())
print('=' * 60)

## Section 3: Data Exploration & Visualization

Before training models, we need to understand our data. Key questions:

1. **Class balance:** Are benign and phishing URLs equally represented?
2. **Feature distributions:** Do features differ between classes?
3. **Feature correlations:** Are any features redundant?

Good exploratory analysis prevents garbage-in, garbage-out.

In [ ]:
# ============================================================
# Cell 7: Visualize Class Distribution and Feature Distributions
# ============================================================
# Check class balance and see how features differ between
# benign and phishing URLs.
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Feature Distributions: Benign vs. Phishing', fontsize=16, fontweight='bold')

# Select the most informative features to visualize
features_to_plot = [
    'url_length', 'num_dots', 'domain_entropy',
    'num_subdomains', 'has_suspicious_keywords', 'uses_https',
]

labels_map = {0: 'Benign', 1: 'Phishing'}
colors = {0: '#2ecc71', 1: '#e74c3c'}

for idx, feature in enumerate(features_to_plot):
    ax = axes[idx // 3][idx % 3]

    for label_val in [0, 1]:
        subset = dataset[dataset['label'] == label_val][feature]
        ax.hist(subset, bins=30, alpha=0.6, label=labels_map[label_val],
                color=colors[label_val], edgecolor='white')

    ax.set_title(feature, fontsize=13, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.show()

# --- Class balance ---
print()
print('=' * 60)
print('CLASS DISTRIBUTION')
print('=' * 60)
class_counts = dataset['label'].value_counts()
for label_val, count in class_counts.items():
    pct = count / len(dataset) * 100
    print(f'  {labels_map[label_val]:10s}: {count:5d} ({pct:.1f}%)')
print('=' * 60)

In [ ]:
# ============================================================
# Cell 8: Correlation Heatmap
# ============================================================
# Visualize how features correlate with each other and with
# the target label. Highly correlated features may be redundant;
# features correlated with 'label' are the most predictive.
# ============================================================

plt.figure(figsize=(12, 10))
corr_matrix = dataset.corr()

# Create a mask for the upper triangle (cleaner visual)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
)
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Print features most correlated with the label
print()
print('=' * 60)
print('FEATURE CORRELATION WITH LABEL (Phishing = 1)')
print('=' * 60)
label_corr = corr_matrix['label'].drop('label').sort_values(ascending=False)
for feature, corr in label_corr.items():
    direction = 'Phishing indicator' if corr > 0 else 'Benign indicator'
    bar = '+' * int(abs(corr) * 20)
    print(f'  {feature:28s}  {corr:+.3f}  {bar:20s}  ({direction})')
print('=' * 60)
print()
print('Key insight: Features with strong positive correlation indicate')
print('phishing; strong negative correlation indicates benign URLs.')

## Section 4: Model Training

### Train/Test Split
We split the data 80/20: 80% for training, 20% for evaluation.
The test set simulates "URLs we have never seen before" -- this is how we measure
 whether the model *generalizes* or just memorizes.

### Two Models
We train two fundamentally different classifiers:

| Model | Type | Strengths | Weaknesses |
|-------|------|-----------|------------|
| **Random Forest** | Ensemble of decision trees | Handles non-linear relationships, robust to outliers, provides feature importance | Can overfit, less interpretable |
| **Logistic Regression** | Linear model | Fast, interpretable, works well when features are linearly separable | Struggles with complex non-linear patterns |

Comparing both helps us understand whether the decision boundary is linear or more complex.

In [ ]:
# ============================================================
# Cell 10: Train/Test Split and Model Training
# ============================================================
# Split data, scale features for Logistic Regression, and
# train both classifiers.
# ============================================================

# Separate features (X) and labels (y)
feature_columns = [col for col in dataset.columns if col != 'label']
X = dataset[feature_columns].values
y = dataset['label'].values

# 80/20 train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

print('=' * 60)
print('TRAIN/TEST SPLIT')
print('=' * 60)
print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'Features:     {X_train.shape[1]}')
print(f'Train class distribution: Benign={sum(y_train==0)}, Phishing={sum(y_train==1)}')
print(f'Test class distribution:  Benign={sum(y_test==0)}, Phishing={sum(y_test==1)}')
print('=' * 60)

# --- Scale features for Logistic Regression ---
# Random Forest does not need scaling, but Logistic Regression
# performs better with standardized features.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Train Random Forest ---
print()
print('Training Random Forest classifier...')
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)
rf_train_acc = rf_model.score(X_train, y_train)
rf_test_acc = rf_model.score(X_test, y_test)
print(f'  Train accuracy: {rf_train_acc:.4f}')
print(f'  Test accuracy:  {rf_test_acc:.4f}')

# --- Train Logistic Regression ---
print()
print('Training Logistic Regression classifier...')
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_SEED,
    solver='lbfgs',
)
lr_model.fit(X_train_scaled, y_train)
lr_train_acc = lr_model.score(X_train_scaled, y_train)
lr_test_acc = lr_model.score(X_test_scaled, y_test)
print(f'  Train accuracy: {lr_train_acc:.4f}')
print(f'  Test accuracy:  {lr_test_acc:.4f}')

print()
print('=' * 60)
print('TRAINING COMPLETE')
print('=' * 60)

## Section 5: Model Evaluation

### Why Accuracy Is Not Enough
In security, **not all errors are equal:**

| Error Type | ML Term | Security Impact |
|-----------|---------|----------------|
| Phishing URL classified as benign | **False Negative** | User visits phishing site, credentials stolen |
| Benign URL classified as phishing | **False Positive** | Legitimate email blocked, user frustrated |

### Key Metrics
- **Precision:** Of all URLs flagged as phishing, how many actually were? (Low precision = alert fatigue)
- **Recall:** Of all actual phishing URLs, how many did we catch? (Low recall = missed attacks)
- **F1 Score:** Harmonic mean of precision and recall (balanced metric)
- **ROC/AUC:** How well the model separates classes across all thresholds

In [ ]:
# ============================================================
# Cell 12: Confusion Matrices
# ============================================================
# The confusion matrix shows exactly where each model makes
# errors: false positives (top-right) and false negatives
# (bottom-left).
# ============================================================

# Generate predictions
rf_preds = rf_model.predict(X_test)
lr_preds = lr_model.predict(X_test_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Random Forest confusion matrix
cm_rf = confusion_matrix(y_test, rf_preds)
ConfusionMatrixDisplay(cm_rf, display_labels=['Benign', 'Phishing']).plot(
    ax=axes[0], cmap='Blues', values_format='d'
)
axes[0].set_title('Random Forest', fontsize=14, fontweight='bold')

# Logistic Regression confusion matrix
cm_lr = confusion_matrix(y_test, lr_preds)
ConfusionMatrixDisplay(cm_lr, display_labels=['Benign', 'Phishing']).plot(
    ax=axes[1], cmap='Oranges', values_format='d'
)
axes[1].set_title('Logistic Regression', fontsize=14, fontweight='bold')

plt.suptitle('Confusion Matrices on Test Set', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# --- Print detailed interpretation ---
print()
print('=' * 60)
print('CONFUSION MATRIX INTERPRETATION')
print('=' * 60)

for name, cm in [('Random Forest', cm_rf), ('Logistic Regression', cm_lr)]:
    tn, fp, fn, tp = cm.ravel()
    print(f'\n  {name}:')
    print(f'    True Negatives  (correctly identified benign):  {tn}')
    print(f'    True Positives  (correctly caught phishing):    {tp}')
    print(f'    False Positives (benign blocked as phishing):   {fp}  <-- Alert fatigue')
    print(f'    False Negatives (phishing missed as benign):    {fn}  <-- DANGER')

print('\n' + '=' * 60)

In [ ]:
# ============================================================
# Cell 13: Classification Reports
# ============================================================
# Detailed precision, recall, and F1 for each model.
# ============================================================

print('=' * 60)
print('CLASSIFICATION REPORT: Random Forest')
print('=' * 60)
print(classification_report(y_test, rf_preds, target_names=['Benign', 'Phishing']))

print('=' * 60)
print('CLASSIFICATION REPORT: Logistic Regression')
print('=' * 60)
print(classification_report(y_test, lr_preds, target_names=['Benign', 'Phishing']))

In [ ]:
# ============================================================
# Cell 14: ROC Curves
# ============================================================
# The Receiver Operating Characteristic (ROC) curve shows
# the tradeoff between True Positive Rate (catching phishing)
# and False Positive Rate (blocking benign) across all
# classification thresholds.
#
# AUC (Area Under Curve) near 1.0 = excellent discrimination.
# ============================================================

# Get probability predictions for ROC
rf_probs = rf_model.predict_proba(X_test)[:, 1]
lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]

# Compute ROC curves
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_probs)
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_probs)

# Compute AUC scores
rf_auc = roc_auc_score(y_test, rf_probs)
lr_auc = roc_auc_score(y_test, lr_probs)

# Plot
plt.figure(figsize=(10, 8))
plt.plot(rf_fpr, rf_tpr, color='#3498db', lw=2.5,
         label=f'Random Forest (AUC = {rf_auc:.3f})')
plt.plot(lr_fpr, lr_tpr, color='#e67e22', lw=2.5,
         label=f'Logistic Regression (AUC = {lr_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random Chance (AUC = 0.500)')

plt.xlabel('False Positive Rate (Benign URLs incorrectly blocked)', fontsize=13)
plt.ylabel('True Positive Rate (Phishing URLs correctly caught)', fontsize=13)
plt.title('ROC Curves: Model Discrimination Ability', fontsize=16, fontweight='bold')
plt.legend(fontsize=12, loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print()
print('=' * 60)
print('ROC / AUC RESULTS')
print('=' * 60)
print(f'  Random Forest AUC:       {rf_auc:.4f}')
print(f'  Logistic Regression AUC: {lr_auc:.4f}')
print()
print('  Interpretation:')
print('    AUC = 1.00 : Perfect classifier')
print('    AUC = 0.50 : No better than random guessing')
print('    AUC > 0.90 : Excellent discrimination')
print('    AUC > 0.80 : Good discrimination')
print('=' * 60)

In [ ]:
# ============================================================
# Cell 15: Feature Importance Analysis
# ============================================================
# Random Forest provides built-in feature importance scores.
# Logistic Regression coefficients show the weight/direction
# of each feature's contribution.
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# --- Random Forest Feature Importance ---
rf_importance = pd.Series(rf_model.feature_importances_, index=feature_columns)
rf_importance = rf_importance.sort_values(ascending=True)

rf_importance.plot(kind='barh', ax=axes[0], color='#3498db', edgecolor='white')
axes[0].set_title('Random Forest: Feature Importance', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Importance (Gini)')

# --- Logistic Regression Coefficients ---
lr_coefs = pd.Series(lr_model.coef_[0], index=feature_columns)
lr_coefs = lr_coefs.sort_values(ascending=True)

colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in lr_coefs]
lr_coefs.plot(kind='barh', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('Logistic Regression: Feature Coefficients', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Coefficient (positive = phishing indicator)')
axes[1].axvline(x=0, color='black', linestyle='-', linewidth=0.5)

plt.suptitle('Which Features Matter Most?', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print()
print('=' * 60)
print('TOP 5 MOST IMPORTANT FEATURES (Random Forest)')
print('=' * 60)
for feat, importance in rf_importance.sort_values(ascending=False).head(5).items():
    bar = '#' * int(importance * 50)
    print(f'  {feat:28s}  {importance:.4f}  {bar}')
print('=' * 60)

## Section 6: Side-by-Side Model Comparison

Let's put both models head-to-head with a comprehensive comparison.

In [ ]:
# ============================================================
# Cell 17: Side-by-Side Model Comparison
# ============================================================
# Compare both models across all key metrics in a single
# summary table and visualization.
# ============================================================

from sklearn.metrics import precision_score, recall_score

# Collect all metrics
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC'],
    'Random Forest': [
        accuracy_score(y_test, rf_preds),
        precision_score(y_test, rf_preds),
        recall_score(y_test, rf_preds),
        f1_score(y_test, rf_preds),
        rf_auc,
    ],
    'Logistic Regression': [
        accuracy_score(y_test, lr_preds),
        precision_score(y_test, lr_preds),
        recall_score(y_test, lr_preds),
        f1_score(y_test, lr_preds),
        lr_auc,
    ],
})

print('=' * 60)
print('MODEL COMPARISON SUMMARY')
print('=' * 60)
print(comparison.to_string(index=False, float_format='{:.4f}'.format))
print('=' * 60)

# --- Grouped bar chart ---
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(comparison['Metric']))
width = 0.35

bars1 = ax.bar(x - width/2, comparison['Random Forest'], width,
               label='Random Forest', color='#3498db', edgecolor='white')
bars2 = ax.bar(x + width/2, comparison['Logistic Regression'], width,
               label='Logistic Regression', color='#e67e22', edgecolor='white')

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)

ax.set_ylabel('Score', fontsize=13)
ax.set_title('Model Performance Comparison', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comparison['Metric'], fontsize=12)
ax.legend(fontsize=12)
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# --- Determine winner ---
print()
rf_f1 = f1_score(y_test, rf_preds)
lr_f1 = f1_score(y_test, lr_preds)
winner = 'Random Forest' if rf_f1 >= lr_f1 else 'Logistic Regression'
print(f'Best overall model (by F1 score): {winner}')
print()
print('Note: In a security context, you may prefer the model with higher')
print('RECALL (fewer missed phishing URLs) even if precision is slightly lower.')

In [ ]:
# ============================================================
# Cell 18: Live Prediction Demo
# ============================================================
# Test the trained model on new URLs to see it in action.
# This demonstrates how the model would be used in production.
# ============================================================

test_urls = [
    'https://www.google.com/search?q=machine+learning',
    'http://192.168.1.100/secure-login/verify.php?token=abc123',
    'https://docs.github.com/en/repositories',
    'http://amaz0n.com.suspicious-tld.tk/account/verify/password',
    'https://www.linkedin.com/in/username',
    'http://login.verify.secure.update.banking.g00gle.xyz/credential',
    'https://en.wikipedia.org/wiki/Machine_learning',
    'http://paypa1.com@malicious-site.tk/login',
]

print('=' * 70)
print('LIVE PREDICTION DEMO')
print('=' * 70)
print()

for url in test_urls:
    # Extract features
    feats = extract_features(url)
    feat_array = np.array([feats[col] for col in feature_columns]).reshape(1, -1)

    # Predict with Random Forest
    rf_pred = rf_model.predict(feat_array)[0]
    rf_conf = rf_model.predict_proba(feat_array)[0][rf_pred]

    # Predict with Logistic Regression (needs scaling)
    feat_scaled = scaler.transform(feat_array)
    lr_pred = lr_model.predict(feat_scaled)[0]
    lr_conf = lr_model.predict_proba(feat_scaled)[0][lr_pred]

    label_map = {0: 'BENIGN', 1: 'PHISHING'}
    rf_label = label_map[rf_pred]
    lr_label = label_map[lr_pred]

    # Truncate long URLs for display
    display_url = url if len(url) <= 65 else url[:62] + '...'

    print(f'  URL: {display_url}')
    print(f'    Random Forest:       {rf_label:10s} (confidence: {rf_conf:.2f})')
    print(f'    Logistic Regression: {lr_label:10s} (confidence: {lr_conf:.2f})')
    print()

print('=' * 70)

## Section 7: Student Exercises

Now it is your turn. Complete the following exercises to deepen your understanding.

---

### Exercise 1: Add New Features
Add **3 new features** to the feature extraction function. Some ideas:
- `has_punycode` -- Does the URL contain "xn--" (internationalized domain encoding)?
- `num_query_params` -- How many query parameters does the URL have?
- `domain_length` -- How long is just the domain portion?
- `has_port_number` -- Does the URL specify a non-standard port?
- `vowel_to_consonant_ratio` -- Random strings have different ratios than words

Retrain the models and see if performance improves.

In [ ]:
# ============================================================
# Exercise 1: Add 3 New Features
# ============================================================
# TODO: Modify the extract_features() function to include
# 3 additional features. Then re-extract features, retrain,
# and compare performance.
#
# Hint: Copy the extract_features function from above, add
# your new features to the dictionary, and re-run the pipeline.
# ============================================================

# Your code here:



### Exercise 2: Try a Different Classifier
Train an additional classifier. Good options:
- `sklearn.svm.SVC` -- Support Vector Machine
- `sklearn.ensemble.GradientBoostingClassifier` -- Gradient Boosted Trees
- `sklearn.neighbors.KNeighborsClassifier` -- K-Nearest Neighbors

Compare its performance to Random Forest and Logistic Regression.

In [ ]:
# ============================================================
# Exercise 2: Try a Different Classifier
# ============================================================
# TODO: Import a new classifier, train it on the same data,
# and generate a classification report + confusion matrix.
#
# Example:
#   from sklearn.ensemble import GradientBoostingClassifier
#   gb_model = GradientBoostingClassifier(random_state=42)
#   gb_model.fit(X_train, y_train)
#   gb_preds = gb_model.predict(X_test)
#   print(classification_report(y_test, gb_preds, ...))
# ============================================================

# Your code here:



### Exercise 3: Experiment with Train/Test Ratio
What happens to model performance when you change the train/test split?
- Try 60/40, 70/30, 90/10
- Does the model overfit with less training data? Underfit with more?

In [ ]:
# ============================================================
# Exercise 3: Experiment with Train/Test Ratios
# ============================================================
# TODO: Try different test_size values in train_test_split
# and observe how model performance changes.
#
# Hint: Loop over different test_size values, retrain, and
# collect F1 scores. Then plot the results.
# ============================================================

# Your code here:



## Section 8: Security Discussion

### False Positives vs. False Negatives: The Security Tradeoff

In production security systems, the cost of errors is **asymmetric:**

| | Cost of False Positive | Cost of False Negative |
|---|---|---|
| **Email Gateway** | Legitimate email goes to spam. User annoyed. | Phishing email delivered. Credentials stolen. |
| **Browser Warning** | User sees warning on safe site. Clicks through. | User visits phishing site with no warning. |
| **SOC Alert** | Analyst wastes 15 minutes investigating. | Breach goes undetected for weeks. |

In most security contexts, **false negatives are far more dangerous** than false positives.
 This means we typically want to **maximize recall** even at the expense of some precision.

You can adjust this tradeoff by changing the classification **threshold:**
- Default threshold: 0.5 (predict phishing if probability > 50%)
- Lower threshold (e.g., 0.3): Catches more phishing (higher recall) but more false alarms
- Higher threshold (e.g., 0.7): Fewer false alarms but misses more phishing

### Adversarial Evasion

Attackers are not static. Once they know what features we use, they can **adapt:**

| Our Feature | Attacker Adaptation |
|------------|--------------------|
| URL length | Use URL shorteners (bit.ly, tinyurl.com) |
| Has IP address | Register cheap domains instead |
| Domain entropy | Use dictionary words in domains |
| Suspicious keywords | Move keywords into path or parameters |
| HTTPS usage | Get free TLS certificates (Let's Encrypt) |

This is the **adversarial ML** problem: the data distribution shifts because attackers
 actively try to evade your model. This is why phishing detection in production requires:
- Continuous retraining on fresh data
- Multiple detection layers (URL + page content + sender reputation)
- Human-in-the-loop for ambiguous cases

### Real-World Deployment Considerations

1. **Latency:** URL classification must happen in milliseconds (before page loads)
2. **Scale:** Billions of URLs per day across an enterprise
3. **Concept drift:** Attacker tactics evolve; models degrade over time
4. **Explainability:** Security analysts need to understand *why* a URL was flagged
5. **False positive management:** Allowlists for known-good domains to reduce noise

## Section 9: Key Takeaways

### What We Learned

1. **Feature engineering is critical.** The raw URL is just a string -- transforming it into
 meaningful numeric features (length, entropy, subdomain count, etc.) is where the real
 intelligence lives.

2. **Different models have different strengths.** Random Forest captures non-linear patterns
 and provides feature importance. Logistic Regression is fast, interpretable, and shows
 directional relationships.

3. **Accuracy is not enough for security.** Precision and recall tell you *how* the model
 fails, not just *how often*. In security, false negatives are usually more costly than
 false positives.

4. **ML is one layer, not the whole solution.** Real-world phishing detection combines
 ML-based URL analysis with blocklists, content inspection, sender reputation, and
 human review.

5. **Attackers adapt.** Models trained on today's phishing URLs will degrade as attackers
 change their techniques. Continuous monitoring and retraining are essential.

### Connection to the Broader Course

This lab demonstrated the **Leverage** theme: using machine learning as a force multiplier
 for defenders. In later labs, we will explore:
- **Attacking AI:** How adversaries can poison training data or evade ML models
- **Defending AI:** How to make ML systems robust against adversarial manipulation
- **Abliteration:** How safety mechanisms in large language models can be bypassed

---

**End of Lab 1**  
**Workshop:** Attacking, Defending & Leveraging Agentic AI  
**Authors:** Derek Banks and Dr. Brian Fehrman